# 03 · Generalization  —  implementation-invariance & multipredicate transfer

- **E1** implementation-invariance — fix the done-*semantics* (item a done), vary its surface code; does the pivot query cluster by *what's done*, not *how written*?
- **E2** multipredicate transfer — done-set × generated-frontier confusion matrix via query transplant; and the pivot-1-quiet / pivot-2-active signature (nothing to subtract at the first pivot).

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")            # so `_diff_primitives` is importable
import torch, numpy as np
import _diff_primitives as P
P.setup("Qwen/Qwen2.5-Coder-32B-Instruct")   # loads model + head masks (once)
from _diff_primitives import *                # helper functions into scope
mt = P.mt
GOAL_HEADS, BOUNDARY_HEADS, UNION_HEADS, GROUP = P.GOAL_HEADS, P.BOUNDARY_HEADS, P.UNION_HEADS, P.GROUP

In [ ]:
# ── multipredicate example: 3 INDEPENDENT predicates, one assignment line each ──
# Each item can be implemented in any order (all orderings correct) — the pivot choice is a
# genuine "which sub-task next", not forced by a data dependency.
ITEMS = {
    "a": ("the list of words longer than three characters", "a = [w for w in words if len(w) > 3]"),
    "b": ("the count of palindromic words",                 "b = sum(w == w[::-1] for w in words)"),
    "c": ("the length of the longest word",                 "c = max(len(w) for w in words)"),
}
ORDER = ["a", "b", "c"]
TASK_SPEC = "\n".join(f"({i+1}) `{k}`: {ITEMS[k][0]}" for i, k in enumerate(ORDER))
QUESTION = ("Write a Python program that implements a function `func` which, given a list of "
            "strings `words`, computes:\n" + TASK_SPEC + "\n")

def build_prefill(done, extra=""):
    """Prefill implementing the assignments in `done` (kept in ORDER), pivot at the next indent."""
    lines = ["<code>", "def func(words: list[str]):"]
    for k in ORDER:
        if k in done:
            lines.append(f"    {ITEMS[k][1]}")
    return "\n".join(lines) + "\n    " + extra   # trailing indent = pivot sits here (about to write next var)

def build_prompt(done, extra=""):
    return get_prompt(QUESTION, build_prefill(done, extra))

def pivot_of(prompt):
    return prepare_input(prompts=prompt, tokenizer=mt)["input_ids"].shape[1] - 1

# task-item token spans are IDENTICAL across done-sets (task precedes the prefill) — compute once.
_p0 = build_prompt([])
ITEM_SPAN = {k: token_span(_p0, ITEMS[k][0]) for k in ORDER}
show_prompt("done={}      ", _p0, ITEM_SPAN["a"])
show_prompt("done={a}     ", build_prompt(["a"]), ITEM_SPAN["b"])
print("item spans:", ITEM_SPAN, " pivot(∅)=", pivot_token(_p0))

## E1 · Implementation-invariance of the query

Item a is done in three surface forms (comprehension / filter / explicit loop); frontier = b in all.
Cosine (over GOAL q-head slices) between `Δq = q(variant) − q({})` across variants.
**Predict:** high pairwise cosine ⇒ the "PROGRESS" the query encodes is the semantic done-set, not tokens.

In [ ]:
A_VARIANTS = {
    "comprehension": "a = [w for w in words if len(w) > 3]",
    "filter":        "a = list(filter(lambda w: len(w) > 3, words))",
    "loop":          "a = []\n    for w in words:\n        if len(w) > 3:\n            a.append(w)",
}
qheads = heads_per_layer(GOAL_HEADS, "q"); Q_LAYERS = sorted(qheads); hd = mt.head_dim
p0 = build_prompt([]); q0 = cache_src_proj(p0, [pivot_of(p0)], Q_LAYERS, "q")

def goal_q_vec(qcache):
    return torch.cat([qcache[l][0, h * hd:(h + 1) * hd] for (l, h) in GOAL_HEADS if l in qcache])

dqs = {}
for name, impl in A_VARIANTS.items():
    p = get_prompt(QUESTION, "<code>\ndef func(words: list[str]):\n    " + impl + "\n    ")
    q = cache_src_proj(p, [pivot_of(p)], Q_LAYERS, "q")
    dqs[name] = goal_q_vec(q_lincomb([(1.0, q), (-1.0, q0)]))

names = list(A_VARIANTS)
tbl = Table(title="E1 · cos(Δq) across surface implementations of a done item", highlight=False)
tbl.add_column("")
for n in names: tbl.add_column(n, justify="right")
for n1 in names:
    tbl.add_row(n1, *[f"{cos(dqs[n1], dqs[n2]):+.3f}" for n2 in names])
rich_print(tbl)
rich_print("[grey58]off-diagonal ≈ 1 ⇒ query invariant to surface implementation (clusters by semantics)[/grey58]")

## E2 · Multipredicate transfer — done-set × frontier confusion matrix

Cache the pivot query for each source done-set, transplant into the ∅-dest, and record which variable
the model writes next. **Predict:** the frontier = one of the *un*done items (e.g. done {a} → b/c;
done {a,b} → c). This is the Spec-1 query transplant with clean combinatorics.

In [ ]:
def q_inject_specs(qcache, dest_pivot):
    qh = heads_per_layer(GOAL_HEADS, "q")
    return ([dict(which="q", cache=qcache, heads=qh, positions=[dest_pivot])],
            [dict(which="q", cache=qcache, heads=qh, positions=[-1])])

dest_tok = prepare_input(prompts=p0, tokenizer=mt); piv0 = pivot_of(p0)
SRC_DONE = [["a"], ["b"], ["c"], ["a", "b"], ["a", "c"], ["b", "c"]]
rich_print(f"[grey70]clean ∅ (no transplant):[/grey70] {escape(repr(greedy(dest_tok, max_new=20)))}")
for done in SRC_DONE:
    p = build_prompt(done)
    q = cache_src_proj(p, [pivot_of(p)], sorted(heads_per_layer(GOAL_HEADS, 'q')), "q")
    pf, dc = q_inject_specs(q, piv0)
    g = greedy(dest_tok, prefill_specs=pf, decode_specs=dc, n_steer=20, max_new=20)
    rich_print(f"src done={{{','.join(done):5s}}} → dest writes: [italic]{escape(repr(g))}[/italic]")

## E2b · Pivot-1-quiet / pivot-2-active signature

At the **first** pivot (∅) there is no PROGRESS to subtract — the choice of which item to do first is a
free priority, not a subtraction. **Predict:** the goal-head write is weaker / less item-selective at ∅
than after ≥1 item is done. Report ‖write‖ and the peak attention onto any single task item.

In [ ]:
tbl = Table(title="E2b · pivot-1 vs pivot-2+ : write-norm and attention selectivity", highlight=False)
tbl.add_column("PROGRESS"); tbl.add_column("‖goal write‖", justify="right")
tbl.add_column("peak attn onto an item", justify="right"); tbl.add_column("argmax item", justify="left")
for done in [[], ["a"], ["a", "b"]]:
    p = build_prompt(done); piv = pivot_of(p)
    info = get_attention_matrices(prepare_input(prompts=p, tokenizer=mt), mt, value_weighted=True)
    per_item = {k: pivot_attn(info, GOAL_HEADS, list(range(*ITEM_SPAN[k])), piv).sum() for k in ORDER}
    top = max(per_item, key=per_item.get)
    tbl.add_row("{" + ",".join(done) + "}", f"{write_norm(p, GOAL_HEADS, piv):.2f}",
                f"{per_item[top]:.3f}", f"`{top}` ({'undone' if top not in done else 'DONE?!'})")
rich_print(tbl)